# Focus Stack Validation and 3D Demo

This notebook treats the 2D x-defocus heatmap and the contrast-through-focus curve as the validation outputs. The 3D surface at the end is kept only as a visual demo because it can exaggerate sparse sweep artifacts.

In [ ]:
import os
import sys

sys.path.insert(0, os.path.abspath(".."))

import matplotlib.pyplot as plt
import numpy as np
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401

from src import constants as C
from src.aerial import aerial_image
from src.dof import focus_stack_contrast
from src.mask import MaskGrid, kirchhoff_mask, line_space_pattern
from src.pupil import PupilSpec

## Study Mask and Focus Sweep

The defocus sweep uses 41 samples so the through-focus trend can be inspected as a 2D validation heatmap rather than inferred from a coarse 3D surface.

In [ ]:
grid = MaskGrid(nx=256, ny=64, pixel_size=2e-9)
mask = kirchhoff_mask(line_space_pattern(grid, pitch_m=40e-9))
defocus_values = np.linspace(-80e-9, 80e-9, 41)
pupil_spec = PupilSpec(grid_size=256, na=C.NA_HIGH, obscuration_ratio=0.0)

samples = focus_stack_contrast(
    mask,
    grid,
    defocus_values,
    pupil_spec=pupil_spec,
    anamorphic=False,
)

print("defocus samples:", len(defocus_values))
print(
    "sampled normalized contrast:",
    [(round(sample.defocus_m * 1e9, 1), round(sample.normalized_contrast, 3)) for sample in samples[::10]],
)

## 2D Validation: Center-Line Intensity Through Focus

Expected behavior: periodic line-space structure along x, strongest contrast near nominal focus, and no NaN/Inf or negative intensity values.

In [ ]:
x_nm = None
focus_lines = []

for dz in defocus_values:
    image, wafer = aerial_image(
        mask,
        grid,
        pupil_spec=PupilSpec(
            grid_size=256,
            na=C.NA_HIGH,
            obscuration_ratio=0.0,
            defocus_m=float(dz),
        ),
        anamorphic=False,
    )
    line = image[wafer.ny // 2, :]
    focus_lines.append(line)
    if x_nm is None:
        x_nm = (np.arange(wafer.nx) - wafer.nx / 2) * wafer.pixel_x_m * 1e9

focus_stack = np.asarray(focus_lines, dtype=np.float64)
focus_stack_norm = focus_stack / max(float(np.max(focus_stack)), 1e-12)
defocus_nm = defocus_values * 1e9

print("focus_stack shape:", focus_stack.shape)
print("focus_stack min/max:", float(np.min(focus_stack)), float(np.max(focus_stack)))
print("focus_stack finite:", bool(np.all(np.isfinite(focus_stack))))
print("defocus samples:", len(defocus_values))
print("x samples:", len(x_nm))

plt.figure(figsize=(9, 5))
plt.imshow(
    focus_stack_norm,
    extent=[x_nm.min(), x_nm.max(), defocus_nm.min(), defocus_nm.max()],
    origin="lower",
    aspect="auto",
)
plt.xlabel("x [nm]")
plt.ylabel("defocus [nm]")
plt.title("2D aerial center-line intensity through focus")
plt.colorbar(label="normalized intensity")
plt.tight_layout()
plt.show()

## 2D Validation: Contrast Through Focus

Contrast is a scalar per defocus slice. Plotting it as a line avoids implying a local x-dependent contrast surface.

In [ ]:
contrast_values = []
for line in focus_stack:
    i_max = float(np.max(line))
    i_min = float(np.min(line))
    contrast = (i_max - i_min) / max(i_max + i_min, 1e-12)
    contrast_values.append(contrast)

contrast_values = np.asarray(contrast_values, dtype=np.float64)
contrast_norm = contrast_values / max(float(np.max(contrast_values)), 1e-12)
best_idx = int(np.argmax(contrast_norm))
best_defocus_nm = float(defocus_nm[best_idx])

print("contrast min/max:", float(np.min(contrast_values)), float(np.max(contrast_values)))
print("contrast finite:", bool(np.all(np.isfinite(contrast_values))))
print("best contrast defocus [nm]:", round(best_defocus_nm, 2))

plt.figure(figsize=(7, 4))
plt.plot(defocus_nm, contrast_norm, marker="o")
plt.axvline(best_defocus_nm, linestyle="--", label=f"best = {best_defocus_nm:.1f} nm")
plt.xlabel("defocus [nm]")
plt.ylabel("normalized contrast")
plt.title("Contrast through focus")
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()

## Demo Only: 3D Focus Surface

This surface reuses the validated 2D stack. Use it for presentation only; use the heatmap and contrast curve above for physical checks.

In [ ]:
X, Z = np.meshgrid(x_nm, defocus_nm)

fig = plt.figure(figsize=(9, 5))
ax = fig.add_subplot(111, projection="3d")
surface = ax.plot_surface(
    X,
    Z,
    focus_stack_norm,
    cmap="viridis",
    linewidth=0,
    antialiased=True,
    alpha=0.9,
)
ax.set_xlabel("x [nm]")
ax.set_ylabel("defocus [nm]")
ax.set_zlabel("normalized intensity")
ax.set_title("Demo only: 3D focus surface; validate with 2D plots above")
fig.colorbar(surface, ax=ax, shrink=0.6, pad=0.1)
fig.tight_layout()
plt.show()